In [3]:
import time, os, threading, cv2
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

In [ ]:
def save_full_screenshot(driver, prefix="bidding_status"):
    try:
        os.makedirs("screenshots", exist_ok=True)
        filename = f"screenshots/{prefix}.png"
        driver.save_screenshot(filename)
        print(f"📸 Screenshot saved: {filename}")
    except Exception as e:
        print(f"❌ Screenshot failed: {e}")
        
        
def scrape(path, headless=False):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--hide-scrollbars")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.get(path)
    driver.maximize_window()
    try:
        provided_u_name = "fourbrotherstrading@icloud.com"
        user_name = WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.ID,'username')))   
        user_name.send_keys(provided_u_name)
    except Exception as e:
        print(f"No username tab found and the error is {e}")
    try:
        provided_pass = "Muhssan7865@"
        password = WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.ID, 'password')))
        driver.execute_script("arguments[0].scrollIntoView();", password)
        password.send_keys(provided_pass)
    except Exception as e:
        print(f"No password tab found and the error is {e}") 
    
    try:
        check = WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.XPATH, './/button[text() = "Sign in"]')))
        if check:
            check.click()
    except Exception as e:
        print(f"No check tab found and the error is {e}")
        

    try:
        popup = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, './/button[@id="start-btn"]')))
        if popup:
            popup.click()
    except Exception as e:
        print("No popup found")
        
        
    return driver


if __name__ == "__main__":
    path = "https://www.ebca.co.uk/portal/auction/buyer/webauction/auction/468"
    driver = scrape(path, headless=False)

    print("Starting data extraction...\n")
    results = []
    previous_title = ""
    previous_last_bid = ""

    try:
        while True:
            if not driver.window_handles:
                print("Browser closed. Exiting loop.")
                break

            details = {}

            try:
                title_el = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, './/div[@class="col-lg-9"]/h1'))
                )
                current_title = title_el.text.strip()
                if current_title != previous_title:
                    previous_title = current_title
                    details["Title"] = current_title
                    try:
                        tables = driver.find_elements(By.XPATH, '//table[@class="table table-condensed vehicle-table"]/tbody')
                        for body in tables:
                            rows = body.find_elements(By.TAG_NAME, 'tr')
                            for tr in rows:
                                lbl = tr.find_element(By.TAG_NAME, 'th').text.strip()
                                val = tr.find_element(By.TAG_NAME, 'td').text.strip()
                                details[lbl] = val
                    except Exception as e:
                        print(f"⚠️ Table parsing error: {e}")

                    reg_value = details.get("Registration", "N/A")
                    details["Registration"] = reg_value

                    try:
                        bidding_prices = driver.find_elements(By.XPATH, '//ul[@id="biddinghistory"]/li')
                        bid_texts = [b.text.strip() for b in bidding_prices if b.text.strip()]
                        details["Bidding History"] = " | ".join(bid_texts)

        
                        for b in bid_texts:
                            if "SOLD" in b.upper():
                                save_full_screenshot(driver, prefix=reg_value)
                                break
                    except Exception as e:
                        print(f"⚠️ Bidding history error: {e}")

            
                    try:
                        current_bid = driver.find_element(By.ID, "currentbid").text.strip()
                        next_bid = driver.find_element(By.ID, "nextbid").text.strip()
                        details["Current Bid"] = f"£{current_bid}"
                        details["Next Bid"] = f"£{next_bid}"
                    except:
                        details["Current Bid"] = ""
                        details["Next Bid"] = ""

                    results.append(details)
                    df = pd.DataFrame(results)
                    df.to_csv("ebca_live.csv", index=False)
                    print(f"✅ Saved data for: {current_title} ({reg_value})")

        
                time.sleep(3)

            except StaleElementReferenceException:
                print("♻️ Stale element, retrying...")
                continue
            except Exception as e:
                print(f"⚠️ Error while extracting data: {e}")

    except KeyboardInterrupt:
        print("🛑 Stopped by user.")
    finally:
        driver.quit()
        print("Browser closed.")

No popup found
Starting data extraction...

⚠️ Error while extracting data: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x12f1213
	0x12f1254
	0x10de6dd
	0x11293a5
	0x112977b
	0x1170382
	0x114b534
	0x116db13
	0x114b2e6
	0x111d321
	0x111e1d4
	0x1545254
	0x154080b
	0x155d0ea
	0x130b118
	0x131311d
	0x12f9518
	0x12f96d9
	0x12e3a68
	0x76715d49
	0x77b4d5db
	0x77b4d561

⚠️ Error while extracting data: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x12f1213
	0x12f1254
	0x10de6dd
	0x11293a5
	0x112977b
	0x1170382
	0x114b534
	0x116db13
	0x114b2e6
	0x111d321
	0x111e1d4
	0x1545254
	0x154080b
	0x155d0ea
	0x130b118
	0x131311d
	0x12f9518
	0x12f96d9
	0x12e3a68
	0x76715d49
	0x77b4d5db
	0x77b4d561

⚠️ Error while extracting data: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x12f1213
	0x12f1254
	0x10de6dd
	0x11293a5
	0x112977b
	0x1170382
	0x114b534
	0x116db13
	0x114b2e6
	0x111d321
	0x111e1d4
	0x1545254
	0x154080b


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x12f1213
	0x12f1254
	0x10de52b
	0x111c635
	0x114b3a6
	0x1146ed1
	0x1146846
	0x10aebfd
	0x10af18e
	0x10af65d
	0x1545254
	0x154080b
	0x155d0ea
	0x130b118
	0x131311d
	0x10ae7ab
	0x10addf7
	0x16916ef
	0x76715d49
	0x77b4d5db
	0x77b4d561


In [ ]:
import pandas as pd
import json
import re

df = pd.read_csv("ebca_live.csv")


df = df[df["Bidding History"].notna() & (df["Bidding History"].astype(str).str.strip() != "")]


bh_index = df.columns.get_loc("Bidding History")
prev_col = df.columns[bh_index - 1]


bidding_data = df["Bidding History"].astype(str).tolist()
prev_col_data = df[prev_col].astype(str).tolist()

if len(bidding_data) >= 2:
    second_last_bh = bidding_data[-2]
    second_last_prev = prev_col_data[-2]


    items = [x.strip() for x in second_last_bh.replace("\n", "|").split("|") if x.strip()]
    
    lots_list = []
    current_lot = None
    current_status = None
    current_bids = []

    for item in items:
        item_clean = item.replace("Â", "") 
        item_lower = item_clean.lower()


        if item_lower.startswith("lot changed"):
            continue


        lot_match = re.search(r"lot\s*(\d+)\s*(sold|provisionally sold)", item_lower)
        if lot_match:

            if current_lot:
                lots_list.append({
                    "Lot": current_lot,
                    "Status": current_status,
                    "Total Bids": sum(current_bids),
                    "Bids": current_bids
                })

            current_lot = lot_match.group(1)
            current_status = lot_match.group(2)
            current_bids = []
            continue


        bid_match = re.search(r"£([\d,]+)", item_clean)
        if bid_match:
            bid_value = int(bid_match.group(1).replace(",", ""))
            current_bids.append(bid_value)


    if current_lot:
        lots_list.append({
            "Lot": current_lot,
            "Status": current_status,
            "Total Bids": sum(current_bids),
            "Bids": current_bids
        })


    output = {
        "Previous Column": second_last_prev,
        "Bidding History by Lots": lots_list
    }


    with open("second_last_bidding.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=4)

    print("✅ Shahi bidding history saved to 'second_last_bidding.json'")

else:
    print("❌ Data me 2 se kam entries hain.")


✅ Shahi bidding history saved to 'second_last_bidding.json'


In [ ]:
import json
import pandas as pd


with open("second_last_bidding.json", "r", encoding="utf-8") as f:
    data = json.load(f)


rows = []
for lot in data["Bidding History by Lots"]:
    rows.append({
        "Lot": lot["Lot"],
        "Bidding Status": lot["Status"],
        "Bidding History": ", ".join([f"£{b:,}" for b in lot["Bids"]]),
        "Last Bid": f"£{max(lot['Bids']):,}" if lot["Bids"] else ""
    })


df = pd.DataFrame(rows)


df.to_csv("second_last_bidding.csv", index=False, encoding="utf-8-sig")

print("✅ CSV created: 'second_last_bidding.csv' (Last Bid = Maximum bid)")


✅ CSV created: 'second_last_bidding.csv' (Last Bid = Maximum bid)


In [ ]:
import pandas as pd


bidding_df = pd.read_csv("second_last_bidding.csv", encoding="utf-8-sig")
ebca_df = pd.read_csv("Ecba_data.csv", encoding="utf-8-sig")


bidding_df['Lot'] = bidding_df['Lot'].astype(str)
ebca_df['Lot'] = ebca_df['Lot'].astype(str)


final_df = pd.merge(ebca_df, bidding_df, on='Lot', how='left')


final_df.to_csv("final_ebca.csv", index=False, encoding="utf-8-sig")

print("✅ Final CSV created: 'final_ebca.csv'")


✅ Final CSV created: 'final_protruck.csv'
